# SafeMine Monitor – Train PPE detector on Google Colab / Kaggle
Runtime → Change runtime type → **GPU (T4)**. Run the cells in order (~45–90 min for 50 epochs with yolo11s).

In [ ]:
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()

## Option A – Roboflow API (free key: roboflow.com → Settings → API key)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_KEY")
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
ds = project.version(30).download("yolov11")
DATA = ds.location + "/data.yaml"
print(DATA)

## Option B – Kaggle dataset already in YOLO format
Upload/attach the dataset, then set `DATA` to its `data.yaml` path (check that train/val/test paths inside the yaml are correct).

In [ ]:
# DATA = "/kaggle/input/<dataset-folder>/data.yaml"

## Train

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.train(data=DATA, epochs=50, imgsz=640, batch=16, patience=15, seed=0, name="ppe_yolo11", plots=True)

## Evaluate on the TEST split (report these numbers in the thesis)

In [ ]:
best = "runs/detect/ppe_yolo11/weights/best.pt"
r = YOLO(best).val(data=DATA, split="test", plots=True)
print(f"P={r.box.mp:.3f} R={r.box.mr:.3f} mAP50={r.box.map50:.3f} mAP50-95={r.box.map:.3f}")

## Download `best.pt` and the result plots
Put `best.pt` into `SafeMine_Monitor/models/best.pt`. Keep `results.png`, `confusion_matrix.png`, `PR_curve.png` for the thesis.

In [ ]:
!zip -r ppe_results.zip runs/detect/ppe_yolo11 runs/detect/val*
from google.colab import files
files.download(best)
files.download("ppe_results.zip")